In [21]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
from openai import OpenAI

In [4]:
df_items = pd.read_json(
    "../data/meta_Electronics_with_category_ratings_100_sample_1000.jsonl",
    lines=True
)

df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Computers,"YQMYXG iPad Air 3rd Gen Apple 10.5"" 2019/2017 ...",4.3,351,[❤【COMPATIBLE WITH】iPad Rotating case is only ...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Review - Smart Case Protector ', '...",YQMYXG,"[Electronics, Computers & Accessories, Tablet ...",{'Standing screen display size': '10.5 Inches'...,B08HDKHDJ5,NaN,NaN,NaN
1,All Electronics,"USB Port Splitter,3 Port USB 2.0 hub Dock [90°...",4.0,514,"[👍【Portable】 mini size, compact, light weight ...","[👍, 3 port usb 2.0 hub Dock [90°/180° Degree R...",6.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Port Splitter', 'url': 'https:...",Adaptermvp,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.94 x 2.36 x 0.39 inc...,B093CX2H8H,NaN,NaN,NaN
2,Computers,"i-UniK Samsung Galaxy Tab E 7.0"" LITE Tablet (...",4.4,335,[Folio design your 2016 Samsung Galaxy TAB E L...,[ONE YEAR REPLACEMENT WARRANTY],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],i-unik,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '7 Inches', '...",B00IGJKCBY,NaN,NaN,NaN
3,All Electronics,"BenQ WXGA LED Business Projector LW500, DLP, F...",4.5,327,"[CRYSTAL CLEAR IMAGES: 2000lm, 20000:1 contras...",[LED meeting room projector WXGA LW500 2000lms],699.00,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'ViewSonic PX Series of Full HD Pro...,BenQ,"[Electronics, Video Projectors]",{'Product Dimensions': '4.69 x 11.65 x 9.02 in...,B0BL992315,NaN,NaN,NaN
4,All Electronics,Remote Control fit for Vizio Home Theater Soun...,4.7,570,[Remote Replacement for Vizio Sound Bar and Vi...,[Sound bar Remote Control Fits for Vizio Sound...,10.95,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'LCD screen don't work.', 'url': 'h...",Elekpia,"[Electronics, Television & Video, Accessories,...",{'Product Dimensions': '5.5 x 1.2 x 0.5 inches...,B09BMFMMR9,NaN,NaN,NaN


In [7]:
list(df_items["features"].items())[0]

(0,
 ['❤【COMPATIBLE WITH】iPad Rotating case is only Designed exclusively for Apple iPad Air (3rd Gen) 10.5 Inch 2019 (Model Number: A2123 / A2152 / A2153 / A2154) and iPad Pro 10.5 Inch 2017 (Model Number: A1701 / A1709 / A1852), Not compatible with any other devices',
  '❤【Function】This iPad 10.5" 2019 protective case allows you to adjust the iPad to 360-degree rotation, and has three built-in design deepening grooves, which can be flexibly viewed horizontally and vertically in the protective case Brings you into the ultimate using experience in multiple landscape and portrait viewing stand angles.',
  '❤【Comprehensive body protection】: Premium synthetic leather exterior, soft microfiber interior lining and hard PC back shell design protect the iPad from impact and falling. Provide enhanced protection for your iPad. This product has an elastic band to better open and close the iPad on the go.',
  '❤[Auto Wake/Sleep Function] There are multiple sleeping magnets inside, which can make y

In [8]:

list(df_items["images"].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/51brL1V8+oL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51brL1V8+oL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/91HxVZKc-rL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51sTnXIeFWL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51sTnXIeFWL._AC_.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/61RmZjWAf+L._AC_SL1000_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51aHMydN8CL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51aHMydN8CL._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/91agCxdx+8L._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51Y45ups2pL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51Y45ups2pL._AC_.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/91RnbLd2LPL._AC_SL1

In [9]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [10]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")


In [18]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [20]:
list(df_items["description"].items())[0]

(0,
 'YQMYXG iPad Air 3rd Gen Apple 10.5" 2019/2017 iPad Pro 10.5" Case-360 Degree Rotating Stand Protective Cover with Auto Wake/Sleep Feature Cover forA2152/A2123/A2153/A2154/A1701/A1709 (Orange) ❤【COMPATIBLE WITH】iPad Rotating case is only Designed exclusively for Apple iPad Air (3rd Gen) 10.5 Inch 2019 (Model Number: A2123 / A2152 / A2153 / A2154) and iPad Pro 10.5 Inch 2017 (Model Number: A1701 / A1709 / A1852), Not compatible with any other devices ❤【Function】This iPad 10.5" 2019 protective case allows you to adjust the iPad to 360-degree rotation, and has three built-in design deepening grooves, which can be flexibly viewed horizontally and vertically in the protective case Brings you into the ultimate using experience in multiple landscape and portrait viewing stand angles. ❤【Comprehensive body protection】: Premium synthetic leather exterior, soft microfiber interior lining and hard PC back shell design protect the iPad from impact and falling. Provide enhanced protection for y

In [19]:
list(df_items["image"].items())[0]

(0, 'https://m.media-amazon.com/images/I/51brL1V8+oL._AC_.jpg')

In [45]:
df_sample = df_items.sample(300, random_state=42)

len(df_sample)

300

In [46]:
data_to_embed = df_sample[
    [
        "description",
        "image",
        "rating_number",
        "price",
        "average_rating",
        "parent_asin",
    ]
].to_dict(orient="records")

data_to_embed

[{'description': 'EasyAcc Case for Samsung Galaxy Tab A 10.1 2019 - Ultra Slim Lightweight Cover with Stand Function Compatible for Samsung Galaxy Tab A T510/ T515 10.1 inch 2019 (Black) Compatible tablet --- made for Samsung Galaxy Tab a 10. 1 2019 (model Number: sm-t510/ sm-t515) (not compatible with any other devices). Ultra slim and lightweight --- a perfect fit for your DEVICE while adding only 5 mm of thickness and offers your device great protection. 2-View angle stand --- stable Trifold stand function, typing and watching are switched easily. Multiple built-in magnets lock together when in stand mode. Precise cutouts --- allow easy access to all ports, buttons and controls, just as the case is not on. Material --- the back cover is made of premium PU Leather and PC, soft microfiber lining and slim shell, The case provides total front-and-back protection against fingerprints, dust and Scratches.',
  'image': 'https://m.media-amazon.com/images/I/41Rrvv3S3sS._AC_.jpg',
  'rating_n

In [30]:
client = OpenAI(
    api_key="lm-studio",
    base_url="http://127.0.0.1:1234/v1"
)

In [33]:
response = client.embeddings.create(
    input="Mohamed",
    model="text-embedding-baai-bge-m3-568m"
)

len(response.data[0].embedding)

1024

In [34]:
def get_embedding(text, model="text-embedding-baai-bge-m3-568m"):
    response = client.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [35]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [36]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-00",
    vectors_config=VectorParams(
        size=len(get_embedding("test")),
        distance=Distance.COSINE,
    ),
)

True

In [47]:
pointstructs = []

for i, data in enumerate(data_to_embed):
    embedding = get_embedding(data["description"])

    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data,
        )
    )

In [48]:
len(pointstructs)

300

In [49]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-00",
    wait=True,
    points=pointstructs,
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [50]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )

    return results

In [51]:
retrieve_data(
    "What kind of pens do you offer?",
    k=10,
).points

[ScoredPoint(id=83, version=2, score=0.5384314, payload={'description': 'One by Wacom Medium Graphics Drawing Tablet & Windows & Drawing Glove, Two-Finger Artist Glove for Drawing Tablet Pen Display, 90% Recycled Material, eco-Friendly, one-Size (1 Pack) Product 1: Certified Works with Chromebook: The only drawing tablet that is certified to work with Chromebook for students, teachers and creators, One by Wacom is simple to use and set-up for any project Product 1: Advanced Electro-Magnetic Pen Technology: Bring Your project and presentations to life with precision with pen technology matching your movement with precision for control and accuracy Product 1: Natural Pen Experience: The included ergonomic 2048 pressure sensitive battery-free pen is responsive and easy to control, giving you the familiar pen-on-paper feel that you are used to Product 1: Perfect Tablet for Software: One by Wacom is a versatile choice for students, artists, and educators with its portability and compatibili